# Fed-SALGATE — Part 3 · Federated Saliency-Consensus Gating (FL experiments)

**Federated Saliency-Consensus Gating for Drone-Based Flood Damage Assessment**

This final part runs the **federated** experiments that define FSCG:

- **Clients = regions**. Part 1's Dirichlet non-IID partitioning gives K=NUM_CLIENTS
  regional clients over the **same FloodNet** domain (real GT train photos).
- Three aggregators on the drone/edge **Lite-UNet** backbone:
  - **FedAvg** (sample-count-weighted averaging)
  - **FedProx** (proximal term penalizing local drift)
  - **Fed-SalGate** (server gates each client update by **spatial saliency consensus**
    — soft-IoU between a client's class-conditional saliency maps and the mean of the
    other clients' maps on a shared probe set; disagreement triggers *localized
    re-sampling* of that client's flood-heavy minority classes).
- **Flagship FSCG model:** the same **FSCG-UNet** learned in Part 2 (hybrid CNN–ViT +
  Spatial Saliency-Consensus Gates) trained under Fed-SalGate — to show the novel model
  beats the baselines in the federated setting too.
- Final evaluation on **held-out REAL GT** (val split from Part 1 — never trained on).

**Inputs:** Part 1 outputs (clients, splits, weights) + Part 2 `model_spec.json`.
**Outputs:** per-round histories, agreements/gates, final comparison table + JSON.

Run on Kaggle free T4 in the background. Attach Part 1 AND Part 2 Outputs as data sources.

## ⚙️ Install packages

In [ ]:
import subprocess, sys

packages = [
    "segmentation-models-pytorch==0.3.3",
    "albumentations==1.3.1",
    "timm",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Packages ready.")

## 📚 Imports

In [ ]:
import os, sys, json, random, warnings, time, pickle, copy, math, tempfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from PIL import Image

import timm

warnings.filterwarnings("ignore")
print("Imports OK. PyTorch", torch.__version__)

In [ ]:
# ===== RunPod real full run (personal Part 3 graph-attention federation experiment) =====
import os
SMOKE_MODE = False

os.environ["PART1_DIR"]     = "/workspace/graphattn/part1_outputs/fed_salgate_outputs"
os.environ["FLOODNET_BASE"] = "/workspace/data/floodnet/FloodNet Challenge - Track 1"
os.environ["SECONDARY_BASE"] = "/workspace/data/floodnet_fixed/data_finally"
os.environ["FSCG_SMOKE"]    = "0"

print("Local override active - SMOKE_MODE =", SMOKE_MODE)
print({k: os.environ[k] for k in ["PART1_DIR", "FLOODNET_BASE", "SECONDARY_BASE", "FSCG_SMOKE"]})


## ⚙️ Config + Part-1 / Part-2 discovery

In [ ]:

SMOKE_TEST  = os.environ.get("FSCG_SMOKE") == "1"
DEBUG_SHORT = os.environ.get("FSCG_DEBUG") == "1" or SMOKE_TEST

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IN_KAGGLE = os.path.isdir("/kaggle/input")
if IN_KAGGLE:
    WORKING_DIR = "/kaggle/working"
else:
    WORKING_DIR = os.path.join(os.path.dirname(os.getcwd()), "working")
    os.makedirs(WORKING_DIR, exist_ok=True)

OUTPUT_DIR = os.path.join(WORKING_DIR, "fed_salgate_outputs")
CKPT_DIR   = os.path.join(WORKING_DIR, "checkpoints")
PLOT_DIR   = os.path.join(OUTPUT_DIR, "plots")
for d in [OUTPUT_DIR, CKPT_DIR, PLOT_DIR, os.path.join(OUTPUT_DIR, "tables")]:
    os.makedirs(d, exist_ok=True)


def _find_outputs_dir(need):
    _pin = "/kaggle/input/models/YOUR_KAGGLE_USERNAME/fedall/tensorflow2/default/1/fed_salgate_outputs"
    if os.path.exists(os.path.join(_pin, "config.json")):
        return _pin
    if os.path.exists(os.path.join(OUTPUT_DIR, "config.json")):
        return OUTPUT_DIR
    if IN_KAGGLE:
        for root, dirs, files in os.walk("/kaggle/input"):
            if all(x in files for x in need):
                if os.path.basename(root) == "fed_salgate_outputs":
                    return root
                return root
    return None


PART1_DIR = os.environ.get("PART1_DIR", "")
if not PART1_DIR:
    _d = _find_outputs_dir(["config.json", "client_indices.pkl", "train_samples.json"])
    PART1_DIR = _d or ""
PART2_DIR = os.environ.get("PART2_DIR", "")
if not PART2_DIR:
    _d = _find_outputs_dir(["part2_centralized_fscg_unet.json", "model_spec.json"])
    PART2_DIR = _d or PART1_DIR or ""
print(f"Part1 dir: {PART1_DIR}")
print(f"Part2 dir: {PART2_DIR}")

if not PART1_DIR or not os.path.exists(os.path.join(PART1_DIR, "config.json")):
    raise FileNotFoundError("Part-1 outputs not found. Attach Part 1's notebook Output.")

with open(os.path.join(PART1_DIR, "config.json")) as f:
    CFG = json.load(f)

NUM_CLASSES   = int(CFG.get("NUM_CLASSES", 10))
CLASS_NAMES   = CFG.get("CLASS_NAMES", ['Background', 'Building-Flooded', 'Building-NonFlooded',
                                        'Road-Flooded', 'Road-NonFlooded', 'Water',
                                        'Tree', 'Vehicle', 'Pool', 'Grass'])
_P2_SPEC = {}
if PART2_DIR and os.path.exists(os.path.join(PART2_DIR, "model_spec.json")):
    with open(os.path.join(PART2_DIR, "model_spec.json")) as f:
        _P2_SPEC = json.load(f)
IMG_SIZE      = int(os.environ.get("FSCG_IMG", str(_P2_SPEC.get("img_size", CFG.get("IMG_SIZE", 320)))))
BATCH_SIZE    = int(os.environ.get("FSCG_BATCH", CFG.get("BATCH_SIZE", 4))) if not SMOKE_TEST else 2
class_weights = CFG.get("class_weights")
NUM_CLIENTS   = int(CFG.get("NUM_CLIENTS", 3))
DIRICHLET_ALPHA = float(CFG.get("DIRICHLET_ALPHA", 0.5))
COLOR_MAP     = {int(k): v for k, v in CFG.get("COLOR_MAP", {}).items()}
print(f"NUM_CLIENTS={NUM_CLIENTS}  IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}")

# ---------------------------------------------------------------------------
# PART-3 v3 settings (all env-overridable to fit one 5h Kaggle/T4 session).
# NOTE: IMAGE/WORKING budgets assume the measured per-round costs from run v2
#       (Lite ~3.7 min/round, FSCG ~6.8 min/round). Defaults sum to ~4.9 h.
# ---------------------------------------------------------------------------
NUM_ROUNDS       = int(os.environ.get("FSCG_FL_ROUNDS", "2" if SMOKE_TEST else "12"))
FL_LOCAL_EPOCHS  = int(os.environ.get("FSCG_FL_EPOCHS", "1" if SMOKE_TEST else "2"))
CLIENT_SUBSAMPLE = int(os.environ.get("FSCG_FL_SUBSAMPLE", "60" if SMOKE_TEST else "100"))
LR               = float(os.environ.get("FSCG_LR", "5e-4"))
PROXIMAL_MU      = float(os.environ.get("FSCG_PROX_MU", "0.01"))
GATE_BETA        = float(os.environ.get("FSCG_GATE_BETA", "4.0"))
GATE_TAU         = float(os.environ.get("FSCG_GATE_TAU", "0.45"))
GATE_MIN, GATE_MAX = 0.5, 1.5
GATE_EMA         = float(os.environ.get("FSCG_GATE_EMA", "0.5"))   # EMA-smooth gates
PROBE_K          = int(os.environ.get("FSCG_PROBE_K", "6" if SMOKE_TEST else "12"))
SAL_CLASSES      = (1, 3, 5)   # Building-Flooded, Road-Flooded, Water
AMP              = os.environ.get("FSCG_AMP", "1") == "1"          # mixed precision
NUM_SEEDS        = int(os.environ.get("FSCG_SEEDS", "2"))          # extra seeds for stats
SEED_ROUNDS      = int(os.environ.get("FSCG_SEED_ROUNDS", "2" if SMOKE_TEST else "4"))
SEED_SUBSAMPLE   = int(os.environ.get("FSCG_SEED_SUBSAMPLE", "30" if SMOKE_TEST else "60"))
ROBUST_ROUNDS    = int(os.environ.get("FSCG_ROBUST_ROUNDS", "2" if SMOKE_TEST else "5"))
DEGRADE_CLIENT   = int(os.environ.get("FSCG_DEGRADE_CLIENT", "1"))
DEGRADE_P        = float(os.environ.get("FSCG_DEGRADE_P", "0.4"))  # label-shuffle prob
_RUN_SEED        = SEED
print(f"FL rounds={NUM_ROUNDS}  local_epochs={FL_LOCAL_EPOCHS}  subsample={CLIENT_SUBSAMPLE}")
print(">>> PART-3 v3: pins fedall discovery | IMG from Part-2 spec | pre-resize cache | deterministic probe")
print(f"gate beta={GATE_BETA}  tau={GATE_TAU}  EMA={GATE_EMA}  probe={PROBE_K}  classes={SAL_CLASSES}  AMP={AMP}")
print(f"seeds={NUM_SEEDS}x({SEED_ROUNDS}r,@subsample{SEED_SUBSAMPLE}) | robustness={ROBUST_ROUNDS}r degraded-client={DEGRADE_CLIENT} p={DEGRADE_P}")


## 🧬 Rebuild datasets + loaders (order identical to Part 1)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

aug_train = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-30, 30), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.RandomGamma(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)

aug_eval = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], is_check_shapes=False)


class FloodSegDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img  = cv2.imread(s["img"]);           img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        aug = self.transform(image=img, mask=mask)
        return aug["image"], aug["mask"].long(), os.path.basename(s["img"])


def load_samples(name):
    with open(os.path.join(PART1_DIR, name)) as f:
        return json.load(f)

with open(os.path.join(PART1_DIR, "client_indices.pkl"), "rb") as f:
    CLIENT_INDICES = pickle.load(f)
with open(os.path.join(PART1_DIR, "gt_val_indices.pkl"), "rb") as f:
    GT_VAL_INDICES = pickle.load(f)

train_samples = load_samples("train_samples.json")
val_samples   = load_samples("val_samples.json")
test_samples  = load_samples("test_samples.json")

# ---- remap Part-1 absolute paths to THIS session's dataset mounts ----
def _find_dataset_roots():
    roots = {}
    fb = os.environ.get("FLOODNET_BASE", "")
    sb = os.environ.get("SECONDARY_BASE", "")
    if fb and os.path.isdir(fb):
        roots["floodnet"] = fb
    if sb and os.path.isdir(sb):
        roots["secondary"] = sb
    if IN_KAGGLE and (not roots.get("floodnet") or not roots.get("secondary")):
        for d in os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else []:
            base = os.path.join("/kaggle/input", d)
            if "floodnet" not in roots:
                cand = os.path.join(base, "FloodNet Challenge - Track 1")
                if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                    roots["floodnet"] = cand
            if "floodnet" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "FloodNet Challenge - Track 1")
                    if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "Train", "Labeled")):
                        roots["floodnet"] = cand
            if "secondary" not in roots:
                cand = os.path.join(base, "floodnet_fixed")
                if os.path.isdir(os.path.join(cand, "data_finally")):
                    roots["secondary"] = cand
            if "secondary" not in roots:
                for sub in os.listdir(base) if os.path.isdir(base) else []:
                    cand = os.path.join(base, sub, "floodnet_fixed")
                    if os.path.isdir(os.path.join(cand, "data_finally")):
                        roots["secondary"] = cand
    return roots

def _remap_path(p, roots):
    if os.path.exists(p):
        return p
    for marker, key in [("/data_finally/", "secondary"),
                        ("/Train/Labeled/", "floodnet"),
                        ("FloodNet Challenge - Track 1", "floodnet")]:
        i = p.find(marker)
        if i != -1 and roots.get(key):
            cand = os.path.join(roots[key], p[i + len(marker):].lstrip("/"))
            if os.path.exists(cand):
                return cand
    return p

_ds_roots = _find_dataset_roots()
_n_remap = 0
for _samples in (train_samples, val_samples, test_samples):
    for _s in _samples:
        for _k in ("img", "mask"):
            _new = _remap_path(_s[_k], _ds_roots)
            if _new != _s[_k]:
                _s[_k] = _new
                _n_remap += 1
print(f"Remapped {_n_remap} Part-1 sample paths to this session's dataset mount "
      f"(floodnet={_ds_roots.get('floodnet')} secondary={_ds_roots.get('secondary')})")

# ---- PART-3 v2: pre-resize cache (same CPU-floor fix as Part 2) ----
try:
    if os.environ.get("FSCG_CACHE", "1") == "1":
        _cache_dir = os.path.join(WORKING_DIR, "resized_cache", str(IMG_SIZE))
        os.makedirs(_cache_dir, exist_ok=True)

        def _resize_to_cache(s, split):
            img = cv2.imread(s["img"])
            msk = cv2.imread(s["mask"], cv2.IMREAD_GRAYSCALE)
            if img is None or msk is None:
                return s
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            stem = os.path.splitext(os.path.basename(s["img"]))[0]
            _ip = os.path.join(_cache_dir, f"{split}_{stem}.jpg")
            _mp = os.path.join(_cache_dir, f"{split}_{stem}_mask.png")
            cv2.imwrite(_ip, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            cv2.imwrite(_mp, msk)
            return {"img": _ip, "mask": _mp}

        _t0c = time.time()
        import concurrent.futures as _cf
        _tasks = ([(s, "tr") for s in train_samples] +
                  [(s, "va") for s in val_samples] +
                  [(s, "te") for s in test_samples])
        _nw = max(1, min(4, os.cpu_count() or 1))
        with _cf.ThreadPoolExecutor(max_workers=_nw) as _ex:
            _out = list(_ex.map(_resize_to_cache,
                                [t[0] for t in _tasks], [t[1] for t in _tasks]))
        _ntr = len(train_samples)
        train_samples = _out[:_ntr]
        val_samples   = _out[_ntr:_ntr + len(val_samples)]
        test_samples  = _out[_ntr + len(val_samples):]
        print(f"Pre-resize cache ready: {len(_tasks)} samples -> {_cache_dir} "
              f"({time.time()-_t0c:.0f}s, {_nw} procs)")
    else:
        print("FSCG_CACHE=0 -> reading full-res originals (slow path)")
except Exception as _ec:
    print("Pre-resize cache FAILED -> continuing with full-res originals:", _ec)


train_dataset  = FloodSegDataset(train_samples, aug_eval)      # no-aug copy (clients add packs at load)
train_aug_ds   = FloodSegDataset(train_samples, aug_train)
val_dataset    = FloodSegDataset(val_samples,   aug_eval)
test_dataset   = FloodSegDataset(test_samples,  aug_eval)

print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")
print(f"client sizes: {[len(c) for c in CLIENT_INDICES]}")
assert sum(len(c) for c in CLIENT_INDICES) == len(train_samples), "client partition mismatch!"

# Held-out GT: full val split is never federated-trained on and never probed (no leakage).
gt_val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2 if not SMOKE_TEST else 0, pin_memory=True)
test_loader    = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2 if not SMOKE_TEST else 0, pin_memory=True)

# Shared saliency-consensus probe set (small, SAME for all clients) -- drawn from
# TRAIN ONLY so the val/test splits stay pristine until the final held-out evaluation.
# Deterministic probe set (aug_eval): saliency agreement must reflect model
# disagreement, not random augmentation noise, so the consensus gate is stable.
probe_idx = list(range(0, len(train_samples), max(1, len(train_samples) // PROBE_K)))[:PROBE_K]
probe_ds   = FloodSegDataset([train_samples[i] for i in probe_idx], aug_eval)
probe_loader = DataLoader(probe_ds, batch_size=1, shuffle=False,
                          num_workers=0, pin_memory=True)
print(f"Probe set: {len(probe_idx)} images from TRAIN (shared, val/test untouched)")
assert max(probe_idx) < len(train_aug_ds), "probe indices must index the train set"
print("Leak guard: probe subset is train-only; val/test remain pristine for final evaluation.")
print("Datasets + loaders ready.")


## 🧠 Models: Lite-UNet (edge) + FSCG-UNet (flagship from Part 2)

In [ ]:
class SaliencyGate(nn.Module):
    """Channel saliency gate (SE-like) -- SPADANet/edge inspired."""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        fc_in = in_channels * 2
        self.fc = nn.Sequential(
            nn.Linear(fc_in, max(4, in_channels // reduction)),
            nn.ReLU(inplace=True),
            nn.Linear(max(4, in_channels // reduction), in_channels),
            nn.Sigmoid())

    def forward(self, x):
        b, c, h, w = x.shape
        g = torch.cat([self.avg_pool(x).view(b, c), self.max_pool(x).view(b, c)], dim=1)
        g = self.fc(g).view(b, c, 1, 1)
        return x * g


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.conv(x)


class LiteUNet(nn.Module):
    """Gap-4 lightweight drone/edge U-Net with saliency gates."""
    def __init__(self, in_channels=3, num_classes=10, features=(16, 32, 64, 128)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.saliency_gates = nn.ModuleList()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        in_ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(in_ch, f))
            self.saliency_gates.append(SaliencyGate(f))
            in_ch = f
        self.bottleneck      = DoubleConv(features[-1], features[-1] * 2)
        self.bottleneck_gate = SaliencyGate(features[-1] * 2)
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))
            self.saliency_gates.append(SaliencyGate(f))
        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skips = []
        nd = len(self.downs)
        for i, (down, gate) in enumerate(zip(self.downs, self.saliency_gates[:nd])):
            x = down(x); x = gate(x); skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        x = self.bottleneck_gate(x)
        skips = skips[::-1]
        gi = nd
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            s = skips[i // 2]
            if x.shape != s.shape:
                x = F.interpolate(x, size=s.shape[2:], mode="bilinear", align_corners=False)
            x = torch.cat((s, x), dim=1)
            x = self.ups[i + 1](x)
            x = self.saliency_gates[gi](x)
            gi += 1
        return self.final_conv(x)

In [ ]:
# ---------- FSCG-UNet (reuse Part 2 architecture) ----------
MODEL_SPEC = {}
if PART2_DIR and os.path.exists(os.path.join(PART2_DIR, "model_spec.json")):
    with open(os.path.join(PART2_DIR, "model_spec.json")) as f:
        MODEL_SPEC = json.load(f)
print("Model spec from Part 2:", MODEL_SPEC)


class AttentionGate(nn.Module):
    def __init__(self, g_channels, x_channels, inter_channels):
        super().__init__()
        self.w_g = nn.Conv2d(g_channels, inter_channels, kernel_size=1)
        self.w_x = nn.Conv2d(x_channels, inter_channels, kernel_size=1)
        self.psi = nn.Conv2d(inter_channels, 1, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, g, x):
        g1 = self.w_g(g); x1 = self.w_x(x)
        return x * self.sigmoid(self.psi(self.relu(g1 + x1)))


class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, rates=(1, 6, 12, 18)):
        super().__init__()
        self.aspp1 = nn.Conv2d(in_channels, out_channels, 1)
        self.aspp2 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[1], dilation=rates[1])
        self.aspp3 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[2], dilation=rates[2])
        self.aspp4 = nn.Conv2d(in_channels, out_channels, 3, padding=rates[3], dilation=rates[3])
        self.global_pool = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                         nn.Conv2d(in_channels, out_channels, 1),
                                         nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(out_channels * 5, out_channels, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        f1, f2, f3, f4 = self.aspp1(x), self.aspp2(x), self.aspp3(x), self.aspp4(x)
        f5 = F.interpolate(self.global_pool(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return self.relu(self.bn(self.out_conv(torch.cat([f1, f2, f3, f4, f5], dim=1))))


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
                                   nn.Linear(int(dim * mlp_ratio), dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        return x + self.mlp(self.norm2(x))


class HybridEncoder(nn.Module):
    def __init__(self, img_size=512, patch_size=4, dim=384, depth=8,
                 backbone_name="convnext_tiny", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          features_only=True, out_indices=(0, 1, 2, 3))
        if "convnext" in backbone_name:
            self.patch_in_ch = 768
            self.skip_channels = [96, 192, 384, 768]
        elif "resnet" in backbone_name:
            self.patch_in_ch = 512
            self.skip_channels = [64, 256, 512, 1024]
        else:
            self.patch_in_ch = 768
            self.skip_channels = [96, 192, 384, 768]
        low_res = max(1, img_size // 32)
        self.patch_size = max(1, min(patch_size, max(1, low_res // 2)))
        self.patch_embed = nn.Conv2d(self.patch_in_ch, dim, kernel_size=self.patch_size, stride=self.patch_size)
        max_patches = max(1, (low_res // self.patch_size) ** 2)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_patches, dim))
        self.blocks = nn.ModuleList([TransformerBlock(dim) for _ in range(depth)])

    def forward(self, x):
        x1, x2, x3, x4 = self.backbone(x)
        pf = self.patch_embed(x4)
        B, C, h, w = pf.shape
        n = h * w
        tokens = pf.flatten(2).transpose(1, 2) + self.pos_embedding[:, :n, :]
        for b in self.blocks:
            tokens = b(tokens)
        return tokens, x1, x2, x3, x4, h, w


class SpatialSaliencyGate(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_channels, in_channels, 3, padding=1),
                                  nn.BatchNorm2d(in_channels), nn.ReLU(inplace=True),
                                  nn.Conv2d(in_channels, 1, 1))

    def forward(self, x):
        m = torch.sigmoid(self.conv(x))
        return x * m, m


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                                nn.Conv2d(in_channels, out_channels, 3, padding=1),
                                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.att_gate = AttentionGate(out_channels, skip_channels, max(8, out_channels // 2)) if skip_channels > 0 else None
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels + (skip_channels if skip_channels > 0 else 0), out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
        self.sal_gate = SpatialSaliencyGate(out_channels)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            skip = F.interpolate(skip, size=x.shape[2:], mode="bilinear", align_corners=False)
            skip = self.att_gate(x, skip) if self.att_gate else skip
            x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x, self.sal_gate(x)


# ------------------------------------------------------------------
# 7b. Graph-Attention + Chebyshev bottleneck (ablation add-on)
# Hand-written in plain PyTorch (no torch_geometric dependency).
# Operates on x4 (native ConvNeXt stage-4 feature, e.g. 10x10=100 nodes
# at 320px input) -- BEFORE the ViT bridge's own patch_embed crushes it
# to a 2x2/4-token grid. Runs as a PARALLEL branch, fused with the
# existing ViT-bridge output; the original ViT path is untouched.
# Graph structure: fixed 4-connectivity spatial grid (same for every
# image, since it's just pixel adjacency on a regular HxW grid).
# ------------------------------------------------------------------
def _build_grid_adjacency(h, w, device):
    """4-connectivity adjacency matrix (with self-loops) for an h x w grid."""
    n = h * w
    adj = torch.zeros(n, n, device=device)
    for i in range(h):
        for j in range(w):
            idx = i * w + j
            adj[idx, idx] = 1.0
            if i > 0:
                adj[idx, (i - 1) * w + j] = 1.0
            if i < h - 1:
                adj[idx, (i + 1) * w + j] = 1.0
            if j > 0:
                adj[idx, i * w + (j - 1)] = 1.0
            if j < w - 1:
                adj[idx, i * w + (j + 1)] = 1.0
    return adj


def _normalized_laplacian_rescaled(adj):
    """Symmetric-normalized graph Laplacian, rescaled to [-1, 1] for Chebyshev."""
    deg = adj.sum(dim=1)
    deg_inv_sqrt = torch.pow(deg.clamp(min=1e-6), -0.5)
    d_mat = torch.diag(deg_inv_sqrt)
    n = adj.shape[0]
    l_sym = torch.eye(n, device=adj.device) - d_mat @ adj @ d_mat
    # For a fixed, bounded grid Laplacian, eigenvalues lie in [0, 2];
    # rescale to [-1, 1] via L_tilde = L - I (lambda_max ~= 2 assumed).
    l_tilde = l_sym - torch.eye(n, device=adj.device)
    return l_tilde


class GraphAttentionLayer(nn.Module):
    """Single-head-per-group GAT layer restricted to a fixed adjacency mask."""
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.1):
        super().__init__()
        assert out_dim % heads == 0
        self.heads = heads
        self.head_dim = out_dim // heads
        self.w = nn.Linear(in_dim, out_dim, bias=False)
        self.attn_src = nn.Parameter(torch.randn(heads, self.head_dim) * 0.02)
        self.attn_dst = nn.Parameter(torch.randn(heads, self.head_dim) * 0.02)
        self.leaky = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(out_dim, out_dim)

    def forward(self, x, adj_mask):
        # x: (B, N, C_in) ; adj_mask: (N, N) with 1 for connected pairs
        B, N, _ = x.shape
        h = self.w(x).view(B, N, self.heads, self.head_dim)          # (B,N,H,d)
        src_score = (h * self.attn_src).sum(-1)                       # (B,N,H)
        dst_score = (h * self.attn_dst).sum(-1)                       # (B,N,H)
        e = self.leaky(src_score.unsqueeze(2) + dst_score.unsqueeze(1))  # (B,N,N,H)
        mask = (adj_mask == 0).view(1, N, N, 1)
        e = e.masked_fill(mask, float("-inf"))
        alpha = torch.softmax(e, dim=2)                                # over neighbors j
        alpha = self.dropout(alpha)
        h_perm = h.permute(0, 2, 1, 3)                                  # (B,H,N,d)
        alpha_perm = alpha.permute(0, 3, 1, 2)                          # (B,H,N,N)
        out = torch.matmul(alpha_perm, h_perm)                          # (B,H,N,d)
        out = out.permute(0, 2, 1, 3).reshape(B, N, self.heads * self.head_dim)
        return self.out_proj(out)


class ChebConvLayer(nn.Module):
    """Chebyshev spectral graph convolution, order K."""
    def __init__(self, in_dim, out_dim, k=3):
        super().__init__()
        self.k = k
        self.theta = nn.ModuleList([nn.Linear(in_dim, out_dim, bias=(i == 0))
                                    for i in range(k)])

    def forward(self, x, l_tilde):
        # x: (B, N, C_in) ; l_tilde: (N, N)
        Tx_0 = x
        out = self.theta[0](Tx_0)
        if self.k > 1:
            Tx_1 = torch.einsum("ij,bjc->bic", l_tilde, x)
            out = out + self.theta[1](Tx_1)
            Tx_prev, Tx_cur = Tx_0, Tx_1
            for i in range(2, self.k):
                Tx_2 = 2 * torch.einsum("ij,bjc->bic", l_tilde, Tx_cur) - Tx_prev
                out = out + self.theta[i](Tx_2)
                Tx_prev, Tx_cur = Tx_cur, Tx_2
        return out


class GraphAttnChebBottleneck(nn.Module):
    """GATConv -> ChebConv over the native (uncrushed) deepest encoder feature."""
    def __init__(self, in_channels, out_channels, heads=4, cheb_k=3):
        super().__init__()
        self.out_channels = out_channels
        self.gat = GraphAttentionLayer(in_channels, out_channels, heads=heads)
        self.norm1 = nn.LayerNorm(out_channels)
        self.cheb = ChebConvLayer(out_channels, out_channels, k=cheb_k)
        self.norm2 = nn.LayerNorm(out_channels)
        self.act = nn.GELU()
        self._adj_cache = {}

    def _get_graph(self, h, w, device):
        key = (h, w, device)
        if key not in self._adj_cache:
            adj = _build_grid_adjacency(h, w, device)
            l_tilde = _normalized_laplacian_rescaled(adj)
            self._adj_cache[key] = (adj, l_tilde)
        return self._adj_cache[key]

    def forward(self, feat_map):
        # feat_map: (B, C, H, W) -- native resolution, NOT yet downsampled by ViT patch_embed
        B, C, H, W = feat_map.shape
        adj, l_tilde = self._get_graph(H, W, feat_map.device)
        x = feat_map.flatten(2).transpose(1, 2)          # (B, N, C)
        x = self.act(self.norm1(self.gat(x, adj)))
        x = self.act(self.norm2(self.cheb(x, l_tilde)))
        return x.transpose(1, 2).reshape(B, self.out_channels, H, W)


class FSCGUNet(nn.Module):
    def __init__(self, num_classes=10, dim=384, depth=8, img_size=512,
                 backbone_name="convnext_tiny", pretrained=True,
                 use_graph_bottleneck=False):
        super().__init__()
        self.img_size = img_size
        self.use_graph_bottleneck = use_graph_bottleneck
        self.encoder = HybridEncoder(img_size=img_size, patch_size=4, dim=dim, depth=depth,
                                     backbone_name=backbone_name, pretrained=pretrained)
        sk = self.encoder.skip_channels
        if self.use_graph_bottleneck:
            self.graph_bottleneck = GraphAttnChebBottleneck(
                in_channels=self.encoder.patch_in_ch, out_channels=dim)
            self.fuse = nn.Conv2d(dim * 2, dim, kernel_size=1)
        self.aspp = ASPP(dim, dim)
        self.dec1 = DecoderBlock(dim, sk[3], 256)
        self.dec2 = DecoderBlock(256, sk[2], 128)
        self.dec3 = DecoderBlock(128, sk[1], 64)
        self.dec4 = DecoderBlock(64,  sk[0], 32)
        self.dec5 = DecoderBlock(32,  0,     32)
        self.head = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        tokens, x1, x2, x3, x4, H, W = self.encoder(x)
        B, N, C = tokens.shape
        feat = tokens.transpose(1, 2).reshape(B, C, H, W)
        if self.use_graph_bottleneck:
            graph_feat = self.graph_bottleneck(x4)
            feat_up = F.interpolate(feat, size=graph_feat.shape[2:],
                                    mode="bilinear", align_corners=False)
            feat = self.fuse(torch.cat([feat_up, graph_feat], dim=1))
        feat = self.aspp(feat)
        f1, _ = self.dec1(feat, x4)
        f2, _ = self.dec2(f1, x3)
        f3, _ = self.dec3(f2, x2)
        f4, _ = self.dec4(f3, x1)
        f5, (_, sal) = self.dec5(f4, None)
        out = F.interpolate(self.head(f5), size=(self.img_size, self.img_size),
                            mode="bilinear", align_corners=False)
        return out, sal


In [ ]:
def _forward(model, x):
    """Return segmentation logits from any of our models."""
    y = model(x)
    if isinstance(y, (tuple, list)):
        return y[0]
    return y


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def make_liteunet(num_classes=NUM_CLASSES):
    return LiteUNet(in_channels=3, num_classes=num_classes)


def make_fscgunet(num_classes=NUM_CLASSES, spec=None):
    spec = spec or {}
    return FSCGUNet(
        num_classes=num_classes,
        dim=int(spec.get("dim", 384)),
        depth=int(spec.get("depth", 8)),
        img_size=int(spec.get("img_size", IMG_SIZE)),
        backbone_name=spec.get("backbone", "convnext_tiny"),
        pretrained=bool(spec.get("pretrained", not SMOKE_TEST)),
    )


def make_fscgunet_graph(num_classes=NUM_CLASSES, spec=None):
    spec = spec or {}
    return FSCGUNet(
        num_classes=num_classes,
        dim=int(spec.get("dim", 384)),
        depth=int(spec.get("depth", 8)),
        img_size=int(spec.get("img_size", IMG_SIZE)),
        backbone_name=spec.get("backbone", "convnext_tiny"),
        pretrained=bool(spec.get("pretrained", not SMOKE_TEST)),
        use_graph_bottleneck=True,
    )


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

m_small = make_liteunet().to(DEVICE)
m_big   = make_fscgunet(spec=MODEL_SPEC).to(DEVICE)
lite_params = count_params(m_small)
fscg_params = count_params(m_big)
print(f"Lite-UNet   params: {lite_params:,} ({lite_params/1e6:.2f} M)")
print(f"FSCG-UNet   params: {fscg_params:,}   ({fscg_params/1e6:.2f} M)")

# sanity forward
m_small.eval()
with torch.no_grad():
    y = _forward(m_small, torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
    print("LiteUNet out:", tuple(y.shape))
m_big.eval()
with torch.no_grad():
    y, sal = m_big(torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
    print("FSCGUNet out:", tuple(y.shape), " sal:", tuple(sal.shape))
del m_small, m_big
torch.cuda.empty_cache()
print("Models ready.")

## 📊 Hyperparameter tracker + parameter budget (Part 3)

In [ ]:

import datetime, platform

hp = {
    "part": 3,
    "reference": "hybrid-cnn-vit-attention-u-net (notebook) strategy",
    "task": "Federated FloodNet segmentation (Fed-SalGate, class-aware gates v3)",
    "num_clients": NUM_CLIENTS,
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "num_rounds": NUM_ROUNDS,
    "local_epochs": FL_LOCAL_EPOCHS,
    "client_subsample": CLIENT_SUBSAMPLE,
    "proximal_mu": PROXIMAL_MU,
    "gate_beta": GATE_BETA,
    "gate_tau": GATE_TAU,
    "gate_min": GATE_MIN,
    "gate_max": GATE_MAX,
    "gate_ema": GATE_EMA,
    "probe_k": PROBE_K,
    "saliency_classes": [int(c) for c in SAL_CLASSES],
    "aggregators": ["FedAvg(lite)", "FedProx(lite, mu=0.01)", "Fed-SalGate(lite)",
                    "Fed-SalGate(FSCG-UNet)"],
    "gating": "per-class saliency-consensus (soft-IoU vs others) + geometric-mean combo + EMA",
    "amp": bool(AMP),
    "seed_study": {"seeds": NUM_SEEDS, "rounds": SEED_ROUNDS, "subsample": SEED_SUBSAMPLE},
    "robustness": {"rounds": ROBUST_ROUNDS, "degraded_client": DEGRADE_CLIENT,
                   "label_shuffle_p": DEGRADE_P},
    "resample_on_disagree": os.environ.get("FSCG_RESAMPLE", "0") == "1",
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "num_classes": NUM_CLASSES,
    "lite_params": int(lite_params),
    "fscg_params": int(fscg_params),
    "optimizer_local": "Adam",
    "loss_local": "CE(class-weighted) + FedProx proximal term",
    "saliency_map": "SSCG guided-sigmoid of dec5 head (no labels involved)",
    "class_weights": ([round(float(w), 4) for w in class_weights]
                      if class_weights is not None else "uniform"),
    "seed": SEED,
    "smoke_test": bool(SMOKE_TEST),
    "python": platform.python_version(),
    "date": datetime.date.today().isoformat(),
}
hp_flat = pd.DataFrame({"hyperparameter": list(hp), "value": [str(v) for v in hp.values()]})
with open(os.path.join(OUTPUT_DIR, "tables", "p3_hyperparams.json"), "w") as f:
    json.dump(hp, f, indent=2, default=str)
hp_flat.to_csv(os.path.join(OUTPUT_DIR, "tables", "p3_hyperparams.csv"), index=False)
print("Part-3 hyperparameters (saved to tables/p3_hyperparams.{json,csv}):")
for k, v in hp.items():
    print(f"  {k:<24} {str(v)[:90]}")


## 📊 Metrics (GPU) + Saliency-consensus scoring

In [ ]:

@torch.no_grad()
def compute_miou(model, loader, num_classes=NUM_CLASSES, max_batches=None):
    model.eval()
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=DEVICE)
    pix_correct = pix_total = 0
    n_batches = 0
    for imgs, msks, _ in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        msks = msks.to(DEVICE, non_blocking=True)
        preds = torch.argmax(_forward(model, imgs), dim=1)
        valid = (msks >= 0) & (msks < num_classes)
        gts = msks[valid]; prs = preds[valid]
        cm += torch.bincount(num_classes * gts + prs, minlength=num_classes ** 2).reshape(
            num_classes, num_classes)
        pix_correct += (preds == msks).sum().item()
        pix_total += msks.numel()
        n_batches += 1
        if max_batches and n_batches >= max_batches:
            break
    inter = torch.diag(cm).float()
    union = cm.sum(dim=1).float() + cm.sum(dim=0).float() - inter
    per_class = torch.where(union > 0, inter / torch.clamp(union, min=1e-8), torch.zeros_like(inter))
    valid = union > 0
    miou = per_class[valid].mean().item() if valid.any() else 0.0
    return miou, {c: float(per_class[c].item()) for c in range(num_classes)}, \
           pix_correct / max(1, pix_total)


@torch.no_grad()
def class_saliency(model, image, classes=SAL_CLASSES):
    """Class-conditional saliency map = max P(class) over the given classes.
    Deterministic (no backprop), cheap, no labels involved."""
    with torch.autocast(device_type="cuda", enabled=AMP and torch.cuda.is_available()):
        logits = _forward(model, image.unsqueeze(0).to(DEVICE))
    prob = F.softmax(logits, dim=1)
    sal = prob[:, list(classes)].max(dim=1).values            # [1,H,W]
    sal = F.interpolate(sal.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    sal = sal.squeeze().cpu().numpy()
    mn, mx = float(np.min(sal)), float(np.max(sal))
    if mx - mn > 1e-9:
        sal = (sal - mn) / (mx - mn)
    return sal.astype(np.float32)


def avg_saliency(model, loader, classes=SAL_CLASSES):
    maps = []
    for imgs, _, _ in loader:
        for i in range(imgs.size(0)):
            s = class_saliency(model, imgs[i], classes=list(classes))
            if s is not None:
                maps.append(s)
    if not maps:
        return None
    return np.mean(np.stack(maps), axis=0)


def soft_iou(a, b, eps=1e-6):
    return float(np.minimum(a, b).sum() / (np.maximum(a, b).sum() + eps))


def _norm_gates(g, tau=GATE_TAU, beta=GATE_BETA, n=None):
    """sigmoid(beta*(agg-tau)), clamp to band, remap so it both (a) never starves a
    client and (b) can still deviate meaningfully from uniform FedAvg weighting."""
    n = n or len(g)
    g = 1.0 / (1.0 + np.exp(-beta * (np.asarray(g, dtype=np.float32) - tau)))
    g = g / (g.sum() + 1e-8) * n
    g = np.clip(g, GATE_MIN, GATE_MAX)
    g = g / (g.sum() + 1e-8) * n
    return g


def saliency_consensus_gates(client_models, probe_loader, beta=GATE_BETA, tau=GATE_TAU,
                             num_clients=NUM_CLIENTS, prev_gates=None,
                             classes=SAL_CLASSES):
    """CLASS-AWARE saliency-consensus gates (v3).

    Novelty: instead of ONE scalar agreement per client, we compute a per-flood-class
    agreement by comparing each client's class-conditional saliency to the mean of the
    OTHER clients (soft-IoU). Class gates are combined with a geometric mean (so a
    client must agree across all flood classes to stay trustable) and then EMA-smoothed
    across rounds to remove the per-round jitter that hurt v2.

    Returns (gates, global_agreements, class_agreements).
    """
    cams = {}
    for cls in classes:
        per = []
        for m in client_models:
            print(f"    [gate-v3] probe saliency for class {cls}...", flush=True)
            cam = avg_saliency(m, probe_loader, classes=[cls])
            per.append(cam if cam is not None else np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32))
        cams[int(cls)] = per

    class_gates, class_agrs = {}, {}
    for cls, per in cams.items():
        agrs, gs = [], []
        for k in range(len(per)):
            others = np.mean([c for i, c in enumerate(per) if i != k], axis=0)
            agrs.append(soft_iou(per[k], others))
        class_agrs[cls] = [float(a) for a in agrs]
        class_gates[cls] = _norm_gates(agrs, tau=tau, beta=beta, n=len(per)).tolist()

    # geometric-mean combination across flood classes
    G = np.ones(num_clients)
    for cls in classes:
        G = G * np.array(class_gates[cls])
    G = G ** (1.0 / max(1, len(classes)))
    gates = _norm_gates(G, tau=tau, beta=beta, n=num_clients)

    if prev_gates is not None:                       # EMA across rounds (kills oscillation)
        gates = GATE_EMA * np.array(prev_gates) + (1 - GATE_EMA) * gates
        gates = gates / (gates.sum() + 1e-8) * num_clients
    gates = _norm_gates(gates, tau=tau, beta=beta, n=num_clients)

    agrs_global = [float(np.mean([class_agrs[cls][k] for cls in classes]))
                   for k in range(num_clients)]
    return gates.tolist(), agrs_global, class_agrs


# Held-out GT evaluator used across experiments (real GT val, never trained/probed on)
def final_eval_gt(model, tag):
    miou, per_class, pix_acc = compute_miou(model, gt_val_loader)
    print(f"  {tag:>30}  mIoU={miou:.4f}  pixel_acc={pix_acc:.4f}")
    return miou, per_class, pix_acc


## 🔄 Federated building blocks (per-client local trains, aggr gates)

In [ ]:

def fed_aggregate(weights_list, client_weights):
    w = torch.tensor(client_weights, dtype=torch.float32, device=DEVICE)
    w = w / (w.sum() + 1e-8)
    g = {}
    for key in weights_list[0].keys():
        st = torch.stack([ww[key].float() for ww in weights_list])
        shape = [len(weights_list)] + [1] * (st.dim() - 1)
        g[key] = (st * w.view(shape)).sum(dim=0)
    return g


def _client_loader(client_idx, round_idx, resample=False):
    """Build a loader for one client (seeded by _RUN_SEED for seed-sweep support)."""
    idx_all = np.array(CLIENT_INDICES[client_idx])
    rng = np.random.RandomState(_RUN_SEED + client_idx * 7919 + round_idx * 104729)
    if len(idx_all) > CLIENT_SUBSAMPLE:
        idx = idx_all[rng.choice(len(idx_all), CLIENT_SUBSAMPLE, replace=False)]
    else:
        idx = idx_all
    sub = Subset(train_aug_ds, idx.tolist())
    nw = 2 if (not SMOKE_TEST and torch.cuda.is_available()) else 0
    if resample:
        weights = []
        for i in range(len(idx)):
            s = train_samples[int(idx[i])]
            mk = np.array(Image.open(s["mask"]))
            if mk.ndim == 3:
                mk = mk[:, :, 0]
            flood_present = np.isin(np.unique(mk), list(SAL_CLASSES)).any()
            weights.append(2.0 if flood_present else 0.5)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        return DataLoader(sub, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=nw, pin_memory=True, drop_last=True)
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=nw, pin_memory=True)


In [ ]:

def _local_train(global_model, client_idx, backbone_fn, local_epochs, lr,
                 proximal_mu=0.0, round_idx=1, resample=False,
                 degrade=False, degrade_p=DEGRADE_P):
    rng = np.random.RandomState(_RUN_SEED + client_idx * 7919 + round_idx * 104729)
    loader = _client_loader(client_idx, round_idx, resample=resample)
    client_model = backbone_fn().to(DEVICE)
    client_model.load_state_dict(copy.deepcopy(global_model.state_dict()))
    opt = optim.Adam(client_model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32,
                                                   device=DEVICE) if class_weights else None)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=AMP and torch.cuda.is_available())
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=AMP and torch.cuda.is_available())
    global_params = list(global_model.parameters())
    client_model.train()
    total_loss = 0.0; ns = 0
    for _ in range(local_epochs):
        for imgs, msks, _ in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            msks = msks.long().to(DEVICE, non_blocking=True)
            if degrade:                                   # simulated faulty client
                r = torch.rand(msks.shape, device=msks.device)
                msks = torch.where(r < degrade_p,
                                   torch.randint_like(msks, 0, NUM_CLASSES), msks)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=AMP and torch.cuda.is_available()):
                logits = _forward(client_model, imgs)
                loss = crit(logits, msks)
                if proximal_mu > 0:
                    prox = 0.0
                    for p, p0 in zip(client_model.parameters(), global_params):
                        prox += ((p - p0.detach()) ** 2).sum()
                    loss = loss + (proximal_mu / 2.0) * prox
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item() * imgs.size(0)
            ns += imgs.size(0)
    return client_model, total_loss / max(1, ns)


def run_federated(name, backbone_fn, aggregator="fedavg",
                  rounds=NUM_ROUNDS, local_epochs=FL_LOCAL_EPOCHS, lr=LR,
                  proximal_mu=PROXIMAL_MU, seed=None,
                  degrade_client=None, degrade_p=DEGRADE_P,
                  probe_loader=probe_loader):
    global _RUN_SEED
    if seed is not None:
        _RUN_SEED = seed
    global_model = backbone_fn().to(DEVICE)
    history = {"round": [], "train_loss": [], "val_miou": [], "gates": [],
               "agreements": [], "class_agreements": [], "resampled": []}
    prev_gates = None

    for round_idx in range(1, rounds + 1):
        local_weights, local_models, local_losses = [], [], []
        for c in range(NUM_CLIENTS):
            resample = (aggregator == "fedsalgate") and round_idx > 1 \
                         and os.environ.get("FSCG_RESAMPLE", "0") == "1"
            degrade = (degrade_client is not None and c == degrade_client)
            cm, al = _local_train(global_model, c, backbone_fn, local_epochs, lr,
                                  proximal_mu if aggregator == "fedprox" else 0.0,
                                  round_idx=round_idx, resample=resample,
                                  degrade=degrade, degrade_p=degrade_p)
            local_weights.append(copy.deepcopy(cm.state_dict()))
            local_models.append(cm)
            local_losses.append(al)

        if aggregator == "fedsalgate":
            gates, agrs, class_agrs = saliency_consensus_gates(
                local_models, probe_loader, prev_gates=prev_gates)
            prev_gates = gates
        else:
            gates, agrs, class_agrs = [1.0 / NUM_CLIENTS] * NUM_CLIENTS, [0.0] * NUM_CLIENTS, {}

        # Saliency-gate snapshot on the FINAL round: keep per-client, per-probe-image
        # saliency maps + consensus so we can visualise the gating mechanism.
        if aggregator == "fedsalgate" and round_idx == rounds:
            _si, _sn = [], []
            for _b in probe_loader:
                for _k in range(_b[0].size(0)):
                    _si.append(_b[0][_k].cpu())
                    _sn.append(_b[2][_k])
            _pc = [[class_saliency(m, img.to(DEVICE)) for img in _si] for m in local_models]
            _cons = np.mean(_pc, axis=0) if _pc else None
            history["snapshot"] = {"imgs": _si, "names": _sn, "per_client": _pc,
                                   "consensus": _cons, "gates": gates, "agreements": agrs}

        global_model.load_state_dict(fed_aggregate(local_weights, gates))

        vm, per_class, _ = compute_miou(global_model, probe_loader)   # fast round log on probe
        history["round"].append(round_idx)
        history["train_loss"].append(float(np.mean(local_losses)))
        history["val_miou"].append(vm)
        history["gates"].append(gates)
        history["agreements"].append(agrs)
        history["class_agreements"].append(class_agrs)
        history["resampled"].append([1 if a < GATE_TAU else 0 for a in agrs])

        for m in local_models:
            del m
        del local_weights
        torch.cuda.empty_cache()

        if round_idx % 5 == 0 or round_idx == 1:
            gs = "|".join(f"{g:.3f}" for g in gates)
            print(f"[{name}] R{round_idx:3d}/{rounds}  loss={history['train_loss'][-1]:.4f}  "
                  f"probe_mIoU={vm:.4f}  gates=[{gs}]")
    return global_model, history


print("Federated training functions ready (v3: class-aware gates + EMA + AMP).")


## 🧪 Experiment 1 — FedAvg / FedProx / Fed-SalGate on Lite-UNet

In [ ]:

t0 = time.time()
lite_fedavg_model, hist_fedavg = run_federated("Lite-UNet/FedAvg", make_liteunet, "fedavg",
                                               seed=SEED)
print(f"FedAvg done in {(time.time()-t0)/60:.1f} min")

t0 = time.time()
lite_fedprox_model, hist_fedprox = run_federated("Lite-UNet/FedProx", make_liteunet, "fedprox",
                                                 seed=SEED)
print(f"FedProx done in {(time.time()-t0)/60:.1f} min")

t0 = time.time()
lite_fedsg_model, hist_fedsg = run_federated("Lite-UNet/Fed-SalGate", make_liteunet, "fedsalgate",
                                             seed=SEED)
print(f"Fed-SalGate(lite) done in {(time.time()-t0)/60:.1f} min")

torch.save(lite_fedavg_model.state_dict(), os.path.join(CKPT_DIR, "lite_fedavg.pth"))
torch.save(lite_fedprox_model.state_dict(), os.path.join(CKPT_DIR, "lite_fedprox.pth"))
torch.save(lite_fedsg_model.state_dict(), os.path.join(CKPT_DIR, "lite_fedsalgate.pth"))
print("Lite-UNet FL models saved.")


## 🧪 Experiment 2 — Flagship: FSCG-UNet under Fed-SalGate

In [ ]:

t0 = time.time()
fscg_fedsg_model, hist_fscg_sg = run_federated("FSCG-UNet/Fed-SalGate", make_fscgunet,
                                               "fedsalgate", rounds=NUM_ROUNDS,
                                               local_epochs=1, lr=LR * 0.1,
                                               proximal_mu=0.0, seed=SEED)
print(f"FSCG-UNet Fed-SalGate done in {(time.time()-t0)/60:.1f} min")
torch.save(fscg_fedsg_model.state_dict(), os.path.join(CKPT_DIR, "fscg_fedsalgate.pth"))
print("FSCG-UNet FL model saved.")


In [ ]:

t0 = time.time()
fscg_graph_fedsg_model, hist_fscg_graph_sg = run_federated(
    "FSCG-UNet+Graph/Fed-SalGate", make_fscgunet_graph,
    "fedsalgate", rounds=NUM_ROUNDS,
    local_epochs=1, lr=LR * 0.1,
    proximal_mu=0.0, seed=SEED)
print(f"FSCG-UNet+Graph Fed-SalGate done in {(time.time()-t0)/60:.1f} min")
torch.save(fscg_graph_fedsg_model.state_dict(), os.path.join(CKPT_DIR, "fscg_graph_fedsalgate.pth"))
print("FSCG-UNet+Graph FL model saved.")



## 🔬 To dissect (3rd run, one session):

1. **Main quartet** (above): FedAvg / FedProx / Fed-SalGate on Lite-UNet + Fed-SalGate on the FSCG-UNet flagship — run 2 already showed **Fed-SalGate ≈ FedAvg**, so:
2. **Seed study** below: FedAvg vs Fed-SalGate (Lite) across `NUM_SEEDS` seeds → **mean ± std** so any claim is statistically defensible.
3. **Robustness**: degrade one client's local labels (`degrade_p=40%` shuffle of a *single* client) → FedAvg should **collapse**, Fed-SalGate's saliency-gate should spot and down-weight the faulty client and **survive**.
4. Every experiment is saved to a checkpoint; the final cell is crash-proof (no more JSON/zip failures).


In [ ]:

# =====================================================================
# SEED STUDY — FedAvg vs Fed-SalGate (LiteUNet), mean ± std over seeds
# =====================================================================
_orig_subsample = CLIENT_SUBSAMPLE
def run_seed_pair(seed, aggregator, name):
    global CLIENT_SUBSAMPLE
    CLIENT_SUBSAMPLE = SEED_SUBSAMPLE          # lighten each seed run
    m, h = run_federated(name, make_liteunet, aggregator,
                         rounds=SEED_ROUNDS, local_epochs=FL_LOCAL_EPOCHS,
                         lr=LR, seed=seed)
    CLIENT_SUBSAMPLE = _orig_subsample
    return m, h

print("=" * 70)
print(f"SEED STUDY: {NUM_SEEDS} seeds, {SEED_ROUNDS} rounds, subsample {SEED_SUBSAMPLE}")
print("=" * 70)
seed_seeds = [SEED + 10 * (i + 1) for i in range(NUM_SEEDS)]
seed_models = {}            # (agg, seed) -> model
seed_finals = []            # rows: [agg, seed, miou, pix_acc]
for seed in seed_seeds:
    for agg, tag in [("fedavg", "FedAvg"), ("fedsalgate", "Fed-SalGate")]:
        m, h = run_seed_pair(seed, agg, f"Lite-UNet/{tag}[s{seed}]")
        seed_models[(agg, seed)] = m
        miou, _pc, pix = final_eval_gt(m, f"Lite-UNet/{tag}[s{seed}]")
        seed_finals.append({"agg": tag, "seed": seed, "miou": miou, "pix": pix})
        torch.save(m.state_dict(), os.path.join(CKPT_DIR, f"lite_{agg}_s{seed}.pth"))

print("=" * 70)
print("SEED-STUDY ROW SUMMARY (held-out real GT, per seed)")
for r in seed_finals:
    print(f"  {r['agg']:<12} seed={r['seed']}  mIoU={r['miou']:.4f}  acc={r['pix']:.4f}")

import statistics as st
_sg_r = [r["miou"] for r in seed_finals if r["agg"] == "Fed-SalGate"]
_sa_r = [r["miou"] for r in seed_finals if r["agg"] == "FedAvg"]
seed_stats = {
    "n_seeds": NUM_SEEDS, "seed_list": seed_seeds, "rounds": SEED_ROUNDS,
    "subsample": SEED_SUBSAMPLE,
    "fedavg_mean": float(st.mean(_sa_r)), "fedavg_std": float(st.stdev(_sa_r)) if len(_sa_r) > 1 else 0.0,
    "fedsalgate_mean": float(st.mean(_sg_r)), "fedsalgate_std": float(st.stdev(_sg_r)) if len(_sg_r) > 1 else 0.0,
    "delta_mean": float(st.mean(_sg_r) - st.mean(_sa_r)),
    "win_vs_fedavg": int(sum(1 for a, b in zip(_sg_r, _sa_r) if b < a) / max(1, len(_sg_r)) * 100),
}
print(f"\nSEED STUDY: FedAvg={seed_stats['fedavg_mean']:.4f}±{seed_stats['fedavg_std']:.4f}  "
      f"Fed-SalGate={seed_stats['fedsalgate_mean']:.4f}±{seed_stats['fedsalgate_std']:.4f}  "
      f"Δ={seed_stats['delta_mean']:+.4f}")


In [ ]:

# =====================================================================
# ROBUSTNESS — one client's labels degraded; does the gate protect us?
# =====================================================================
print("=" * 70)
print(f"ROBUSTNESS: {ROBUST_ROUNDS} rounds; degrade client {DEGRADE_CLIENT} "
      f"(p={DEGRADE_P} label shuffle). FedAvg vs Fed-SalGate (Lite-UNet).")
print("=" * 70)
rob_suffix = 1000003
rob_avg, hist_rob_avg = run_federated("Lite-UNet/FedAvg[rob]", make_liteunet, "fedavg",
                                      rounds=ROBUST_ROUNDS, local_epochs=FL_LOCAL_EPOCHS,
                                      lr=LR, seed=SEED + rob_suffix,
                                      degrade_client=DEGRADE_CLIENT, degrade_p=DEGRADE_P)
rob_sg, hist_rob_sg = run_federated("Lite-UNet/Fed-SalGate[rob]", make_liteunet, "fedsalgate",
                                    rounds=ROBUST_ROUNDS, local_epochs=FL_LOCAL_EPOCHS,
                                    lr=LR, seed=SEED + rob_suffix,
                                    degrade_client=DEGRADE_CLIENT, degrade_p=DEGRADE_P)
rob_avg_miou, _pc, rob_avg_pix = final_eval_gt(rob_avg, "Lite-UNet/FedAvg[degraded]")
rob_sg_miou, _pc, rob_sg_pix = final_eval_gt(rob_sg, "Lite-UNet/Fed-SalGate[degraded]")
torch.save(rob_avg.state_dict(), os.path.join(CKPT_DIR, "lite_fedavg_rob.pth"))
torch.save(rob_sg.state_dict(), os.path.join(CKPT_DIR, "lite_fedsalgate_rob.pth"))

robust_stats = {
    "rounds": ROBUST_ROUNDS, "degraded_client": DEGRADE_CLIENT, "degrade_p": DEGRADE_P,
    "fedavg_miou": float(rob_avg_miou), "fedsalgate_miou": float(rob_sg_miou),
    "fedavg_pix_acc": float(rob_avg_pix), "fedsalgate_pix_acc": float(rob_sg_pix),
    "margin": float(rob_sg_miou - rob_avg_miou),
}
print("\nROBUSTNESS (one degraded client):")
print(f"  FedAvg       mIoU={rob_avg_miou:.4f}  (expected collapse)")
print(f"  Fed-SalGate  mIoU={rob_sg_miou:.4f}  (expect survival via gating)")
print(f"  Margin Δ={rob_sg_miou - rob_avg_miou:+.4f}")


In [ ]:

# ---- thesis figures for seed study + robustness (43-46) ----
# (43) seed scatter / error-bar comparison
if seed_finals:
    xs_a = [r["seed"] for r in seed_finals if r["agg"] == "FedAvg"]
    ys_a = [r["miou"] for r in seed_finals if r["agg"] == "FedAvg"]
    xs_g = [r["seed"] for r in seed_finals if r["agg"] == "Fed-SalGate"]
    ys_g = [r["miou"] for r in seed_finals if r["agg"] == "Fed-SalGate"]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(xs_a, ys_a, s=90, marker="o", color="#7f9cc9", label="FedAvg", zorder=3)
    ax.scatter(xs_g, ys_g, s=90, marker="D", color="#c977a2", label="Fed-SalGate", zorder=3)
    for x, y in zip(xs_a, ys_a): ax.text(x + 6, y, f"{y:.3f}", fontsize=8)
    for x, y in zip(xs_g, ys_g): ax.text(x + 6, y, f"{y:.3f}", fontsize=8)
    ax.axhline(seed_stats["fedavg_mean"], color="#7f9cc9", ls=":", lw=1.2)
    ax.axhline(seed_stats["fedsalgate_mean"], color="#c977a2", ls=":", lw=1.2)
    ax.set_xlabel("Seed"); ax.set_ylabel("Held-out GT mIoU")
    ax.set_title(f"Seed study — FedAvg vs Fed-SalGate (mean±std, n={NUM_SEEDS})")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "43_fed_seed_study.png"), dpi=150, bbox_inches="tight"); plt.show()
    print("43_fed_seed_study.png saved.")



# (44) robustness figure: FedAvg collapses, Fed-SalGate survives a degraded client
if robust_stats:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
    names = ["FedAvg", "Fed-SalGate"]
    vals = [robust_stats["fedavg_miou"], robust_stats["fedsalgate_miou"]]
    barl = ax1.bar(names, vals, color=["#e29b62", "#6fbf93"])
    for b, v in zip(barl, vals):
        ax1.text(b.get_x() + b.get_width()/2, v + 0.004, f"{v:.4f}", ha="center", fontsize=10)
    ax1.set_ylabel("Held-out GT mIoU")
    ax1.set_ylim(0, max(vals) * 1.5)
    ax1.set_title(f"Robustness — client {robust_stats['degraded_client']} degraded\n(p={robust_stats['degrade_p']:.2f} label shuffle)")
    ax1.grid(alpha=0.3, axis="y")

    # gate trajectory on the degraded client
    h = hist_rob_sg
    G = np.array(h["gates"])                 # [rounds, clients]
    ax2.plot(h["round"], G[:, DEGRADE_CLIENT], "o-", color="#d62728", lw=2,
             label=f"degraded client {DEGRADE_CLIENT} gate")
    for k in range(G.shape[1]):
        if k != DEGRADE_CLIENT:
            ax2.plot(h["round"], G[:, k], "o-", color="#7f7f7f", lw=1.2,
                     label=f"healthy client {k} gate")
    ax2.axhline(1.0 / NUM_CLIENTS, color="k", ls=":", lw=1, label="FedAvg weight")
    ax2.set_xlabel("Round"); ax2.set_ylabel("Gate weight (λ)")
    ax2.set_title("Fed-SalGate down-weights the faulty client via saliency consensus")
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "44_fed_robustness.png"), dpi=150, bbox_inches="tight"); plt.show()
    print("44_fed_robustness.png saved.")

print("\nDONE seed + robustness experiments.")


## 🏁 Final evaluation on held-out REAL GT (val, never trained on)

In [ ]:
def final_eval(model, loader, tag):
    model.eval()
    miou, per_class, pix_acc = compute_miou(model, loader)
    print(f"  {tag:>26}  mIoU={miou:.4f}  pixel_acc={pix_acc:.4f}")
    return miou, per_class, pix_acc

print("=" * 70)
print("FINAL EVALUATION -- held-out REAL GT (val split, never trained on)")
print("=" * 70)
_best_by_hist = {"LiteUNet / FedAvg": hist_fedavg, "LiteUNet / FedProx": hist_fedprox,
                 "LiteUNet / Fed-SalGate": hist_fedsg, "FSCG-UNet / Fed-SalGate": hist_fscg_sg}
def _best_round(tag, h):
    return max(h["val_miou"]) if h["val_miou"] else 0.0
finals = {}
for tag, model in [("LiteUNet / FedAvg", lite_fedavg_model),
                   ("LiteUNet / FedProx", lite_fedprox_model),
                   ("LiteUNet / Fed-SalGate", lite_fedsg_model),
                   ("FSCG-UNet / Fed-SalGate", fscg_fedsg_model)]:
    miou, per_class, pix_acc = final_eval(model, gt_val_loader, tag)
    br = _best_round(tag, _best_by_hist[tag])
    finals[tag] = {"heldout_miou": miou, "pixel_acc": pix_acc, "per_class": per_class,
                   "best_round_miou": float(br)}
_dm_fa = finals["LiteUNet / FedAvg"]["heldout_miou"]
_dm_fp = finals["LiteUNet / FedProx"]["heldout_miou"]
_dm_sg = finals["LiteUNet / Fed-SalGate"]["heldout_miou"]
_dm_bg = finals["FSCG-UNet / Fed-SalGate"]["heldout_miou"]
deltas_main = {"vs_fedavg": float(_dm_sg - _dm_fa),
               "vs_fedprox": float(_dm_sg - _dm_fp),
               "fscg_vs_avg": float(_dm_bg - _dm_fa)}
print("deltas_main:", deltas_main)


In [ ]:

# ---- real held-out evaluation for the graph-attention federated model ----
_graph_miou, _graph_per_class, _graph_pix_acc = final_eval(
    fscg_graph_fedsg_model, gt_val_loader, "FSCG-UNet+Graph / Fed-SalGate")
_graph_best_round = max(hist_fscg_graph_sg["val_miou"]) if hist_fscg_graph_sg["val_miou"] else 0.0
_graph_result = {
    "heldout_miou": float(_graph_miou),
    "pixel_acc": float(_graph_pix_acc),
    "per_class_iou": {str(k): v for k, v in _graph_per_class.items()},
    "best_round_miou": float(_graph_best_round),
}
with open(os.path.join(OUTPUT_DIR, "fscg_graph_fedsalgate_result.json"), "w") as f:
    json.dump(_graph_result, f, indent=2)
print("Saved fscg_graph_fedsalgate_result.json:", _graph_result["heldout_miou"], _graph_result["pixel_acc"])


## 📊 Comparison table + plots

In [ ]:
rows = []
for tag in ["LiteUNet / FedAvg", "LiteUNet / FedProx", "LiteUNet / Fed-SalGate",
            "FSCG-UNet / Fed-SalGate"]:
    f = finals[tag]
    rows.append([tag, f"{f['heldout_miou']:.4f}", f"{f['pixel_acc']:.4f}"])

sg = finals["LiteUNet / Fed-SalGate"]["heldout_miou"]
fa = finals["LiteUNet / FedAvg"]["heldout_miou"]
fp = finals["LiteUNet / FedProx"]["heldout_miou"]
big = finals["FSCG-UNet / Fed-SalGate"]["heldout_miou"]
rows.append(["---", "---", "---"])
rows.append(["Fed-SalGate vs FedAvg (LiteUNet ΔmIoU)", f"{sg - fa:+.4f}", ""])
rows.append(["Fed-SalGate vs FedProx (LiteUNet ΔmIoU)", f"{sg - fp:+.4f}", ""])
rows.append(["FSCG-UNet/FedSalGate vs LiteUNet/FedAvg Δ", f"{big - fa:+.4f}", ""])

fig, ax = plt.subplots(figsize=(13, 5.2))
ax.axis("off")
table = ax.table(cellText=rows, colLabels=["Model", "Held-out GT mIoU", "Pixel Acc"],
                 cellLoc="center", loc="center")
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.2, 1.6)
for i in range(len(rows)):
    if rows[i][0].startswith("FSCG-UNet") or rows[i][0].startswith("LiteUNet / Fed-SalGate"):
        for j in range(3):
            table[(i + 1, j)].set_facecolor("#dff0d8")
plt.title("Fed-SALGATE Part 3 -- Held-out GT RESULTS (real FloodNet)", fontsize=15, fontweight="bold", pad=18)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "21_fed_results_table.png"), dpi=140, bbox_inches="tight")
plt.show()
print("21_fed_results_table.png saved.")

# convergence curves
fig, ax = plt.subplots(figsize=(12, 5))
for h, tag in [(hist_fedavg, "FedAvg"), (hist_fedprox, "FedProx"),
               (hist_fedsg, "Fed-SalGate"), (hist_fscg_sg, "FSCG-UNet/Fed-SalGate")]:
    ax.plot(h["round"], h["val_miou"], "o-", label=tag)
ax.set_xlabel("Round"); ax.set_ylabel("Probe mIoU")
ax.set_title("Federated convergence (round-log on shared probe set)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "22_fed_convergence.png"), dpi=140, bbox_inches="tight")
plt.show()
print("22_fed_convergence.png saved.")

# gates / agreements over rounds
if any(h["agreements"] for h in [hist_fedsg, hist_fscg_sg]):
    fig, axes = plt.subplots(2, 1, figsize=(11, 7))
    for h, tag, ax in [(hist_fedsg, "Lite-UNet/Fed-SalGate", axes[0]),
                       (hist_fscg_sg, "FSCG-UNet/Fed-SalGate", axes[1])]:
        agrs = np.array(h["agreements"])
        gates = np.array(h["gates"])
        for k in range(min(agrs.shape[1], 5)):
            ax.plot(h["round"], agrs[:, k], marker=".", label=f"agreement client {k}")
            ax.axhline(GATE_TAU, color="r", ls="--", lw=0.8)
        ax.set_title(f"{tag} -- saliency agreements (red = gate threshold {GATE_TAU})")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "23_fed_agreements.png"), dpi=140, bbox_inches="tight")
    plt.show()
    print("23_fed_agreements.png saved.")

## 🌡️ Fed-SalGate diagnostics — agreement heatmaps, gate weights, per-class comparisons

In [ ]:
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

# (1) agreement heatmap: rounds x clients (both Fed-SalGate runs)
for h, tag, fname in [(hist_fedsg, "Lite-UNet / Fed-SalGate", "24_fed_agreement_heatmap.png"),
                      (hist_fscg_sg, "FSCG-UNet / Fed-SalGate", "25_fed_agreement_heatmap_big.png")]:
    A = np.array(h["agreements"])
    if A.size == 0:
        continue
    fig, ax = plt.subplots(figsize=(11, max(4, 0.5 * A.shape[0])))
    im = ax.imshow(A, aspect="auto", cmap="YlGnBu", vmin=0.0, vmax=1.0)
    ax.set_yticks(range(A.shape[0]))
    ax.set_yticklabels([f"R{r}" for r in h["round"]], fontsize=8)
    ax.set_xticks(range(A.shape[1]))
    ax.set_xticklabels([f"C{c}" for c in range(A.shape[1])], fontsize=8)
    ax.set_ylabel("Round"); ax.set_xlabel("Client")
    ax.set_title(f"{tag} -- saliency-agreement score per round (threshold τ={GATE_TAU})")
    plt.colorbar(im, ax=ax); plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=140, bbox_inches="tight")
    plt.show()
    print(f"{fname} saved.")

# (2) gate weights over rounds
for h, tag, fname in [(hist_fedsg, "Lite-UNet / Fed-SalGate", "26_fed_gate_weights.png"),
                      (hist_fscg_sg, "FSCG-UNet / Fed-SalGate", "27_fed_gate_weights_big.png")]:
    G = np.array(h["gates"])
    if G.size == 0:
        continue
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for k in range(G.shape[1]):
        ax.plot(h["round"], G[:, k], "o-", label=f"client {k}")
    ax.axhline(1.0, color="k", ls=":", lw=1.2)
    ax.set_xlabel("Round"); ax.set_ylabel("Gate weight (λ)")
    ax.set_title(f"{tag} -- per-client aggregation gate weights")
    ax.legend(fontsize=8, ncol=max(1, NUM_CLIENTS // 5)); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=140, bbox_inches="tight")
    plt.show()
    print(f"{fname} saved.")

# (3) per-class IoU grouped bars (4 federated models, flood classes highlighted)
model_tags = ["LiteUNet / FedAvg", "LiteUNet / FedProx", "LiteUNet / Fed-SalGate",
              "FSCG-UNet / Fed-SalGate"]
flood = [c in SAL_CLASSES for c in range(NUM_CLASSES)]
colors = ["#7f9cc9", "#e29b62", "#c977a2", "#6fbf93"]
fig, ax = plt.subplots(figsize=(14, 5.2))
xs = np.arange(NUM_CLASSES); w = 0.9 / len(model_tags)
for k, tag in enumerate(model_tags):
    ious = [finals[tag]["per_class"].get(c, 0.0) for c in range(NUM_CLASSES)]
    ax.bar(xs + (k - 1.5) * w, ious, width=w, label=tag, color=colors[k])
for c in range(NUM_CLASSES):
    if flood[c]:
        ax.axvspan(c - 0.5, c + 0.5, color="gold", alpha=0.18)
ax.set_xticks(xs); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_ylabel("IoU"); ax.set_title("Federated models -- per-class IoU on held-out GT  (gold = flood classes)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "28_fed_perclass_iou.png"), dpi=140, bbox_inches="tight")
plt.show()
print("28_fed_perclass_iou.png saved.")

# (4) models x classes heatmap
arr = np.zeros((len(model_tags), NUM_CLASSES))
for k, tag in enumerate(model_tags):
    arr[k] = [finals[tag]["per_class"].get(c, 0.0) for c in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(12, 4.2))
im = ax.imshow(arr, cmap="YlGnBu", vmin=0, vmax=0.95)
ax.set_yticks(range(len(model_tags))); ax.set_yticklabels(model_tags, fontsize=8)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)
for k in range(arr.shape[0]):
    for c in range(arr.shape[1]):
        ax.text(c, k, f"{arr[k, c]:.2f}", ha="center", va="center", fontsize=6,
                color="black")
ax.set_title("Models x classes IoU (held-out GT)")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "29_fed_model_class_heatmap.png"), dpi=140, bbox_inches="tight")
plt.show()
print("29_fed_model_class_heatmap.png saved.")

# (5) radar chart on flood-sensitive axes
radar_axes = ["mIoU"] + [CLASS_NAMES[c] for c in SAL_CLASSES] + ["pixel acc"]
ang = np.linspace(0, 2 * np.pi, len(radar_axes), endpoint=False).tolist()
ang += ang[:1]
fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True))
for k, tag in enumerate(model_tags):
    vals = [finals[tag]["heldout_miou"]] + \
           [finals[tag]["per_class"].get(c, 0.0) for c in SAL_CLASSES] + \
           [finals[tag]["pixel_acc"]]
    vals += vals[:1]
    ax.plot(ang, vals, "o-", label=tag, color=colors[k], lw=2)
    ax.fill(ang, vals, alpha=0.05, color=colors[k])
ax.set_xticks(ang[:-1]); ax.set_xticklabels(radar_axes, fontsize=9)
ax.set_ylim(0, 1.0); ax.set_title("Flood-sensitive comparison radar", pad=24)
ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.05), fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "30_fed_radar.png"), dpi=140, bbox_inches="tight")
plt.show()
print("30_fed_radar.png saved.")

# (6) gains bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))
gains = [("Fed-SalGate vs FedAvg", sg - fa),
         ("Fed-SalGate vs FedProx", sg - fp),
         ("FSCG-UNet/FedSalGate vs LiteUNet/FedAvg", big - fa)]
labels_g = [g[0] for g in gains]; vals_g = [g[1] for g in gains]
bars = ax.bar(labels_g, vals_g, color=["#6fbf93", "#7f9cc9", "#c977a2"])
for b, v in zip(bars, vals_g):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.002 * np.sign(v), f"{v:+.4f}",
            ha="center", fontsize=10)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("Δ mIoU"); ax.set_title("FSCG gains over FL baselines (held-out GT)")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "31_fed_gains.png"), dpi=140, bbox_inches="tight")
plt.show()
print("31_fed_gains.png saved.")

# (7) CSV exports
rows = []
for tag in model_tags:
    rows.append({"model": tag,
                 "heldout_miou": finals[tag]["heldout_miou"],
                 "pixel_acc": finals[tag]["pixel_acc"],
                 **{f"iou_{CLASS_NAMES[c]}": finals[tag]["per_class"].get(c, 0.0)
                    for c in range(NUM_CLASSES)}})
pd.DataFrame(rows).set_index("model").to_csv(os.path.join(TABLE_DIR, "p3_fed_per_class_iou.csv"))
pd.DataFrame([{"comparison": g, "delta_miou": v} for g, v in gains]
             ).to_csv(os.path.join(TABLE_DIR, "p3_fed_gains.csv"), index=False)
print("Saved tables: p3_fed_per_class_iou.csv, p3_fed_gains.csv")

## 🖼️ Qualitative validation — sample grids, saliency-gate visual, confusion matrices, error maps
These figures directly evidence *what Fed-SalGate changes qualitatively*: held-out REAL-test sample
predictions (≥6 images per model), a cross-model side-by-side, the actual saliency maps the gating
consensus is built from, per-class confusion matrices, and FP/FN error overlays. All on data the FL
models never optimised on (final models are used read-only here).

In [ ]:
# ------------------------- qualitative validation (thesis figures) -------------------------
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
def _denorm(t):  # t: [3,H,W] float tensor, ImageNet-normalised (matches aug_eval)
    a = t.permute(1, 2, 0).cpu().numpy()
    return np.clip(a * STD + MEAN, 0, 1)

def _grab(loader, n=8):
    imgs, msks, names = [], [], []
    for b in loader:
        for k in range(b[0].size(0)):
            imgs.append(b[0][k]); msks.append(b[1][k]); names.append(b[2][k])
            if len(imgs) >= n:
                return imgs, msks, names
    return imgs, msks, names

def _pred(model, img):
    return torch.argmax(_forward(model, img.unsqueeze(0).to(DEVICE)), dim=1)[0].cpu().numpy()

def plot_sample_grid(model, tag, imgs, msks, names, fname, n_show=10):
    n = min(n_show, len(imgs))
    fig, axes = plt.subplots(n, 3, figsize=(13, 4.2 * n))
    if n == 1: axes = axes[None, :]
    for i in range(n):
        p = _pred(model, imgs[i])
        axes[i, 0].imshow(_denorm(imgs[i])); axes[i, 0].set_title(f"Image - {names[i]}", fontsize=8); axes[i, 0].axis("off")
        axes[i, 1].imshow(msks[i].numpy(), cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 1].set_title("GT", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(p, cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 2].set_title("Prediction", fontsize=9); axes[i, 2].axis("off")
    fig.suptitle(f"Held-out REAL-test samples — {tag}  (mIoU {finals[tag]['heldout_miou']:.3f})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fname), dpi=150, bbox_inches="tight"); plt.show()
    print(fname, "saved.")

# sample grids (>=6 images each) on held-out real test set
# grab 10 held-out REAL-test samples and rank them by flood-coverage (descending)
# so the showcase leads with the most flood-rich scenes (flood = SAL_CLASSES).
_ti, _tm, _tn = _grab(test_loader, n=10)
def _flood_frac(mask_np):
    tot = mask_np.size
    return float(np.isin(mask_np, list(SAL_CLASSES)).sum()) / max(1, tot)
_order = sorted(range(len(_ti)), key=lambda k: _flood_frac(_tm[k].numpy()), reverse=True)
_ti = [_ti[k] for k in _order]; _tm = [_tm[k] for k in _order]; _tn = [_tn[k] for k in _order]
print("Sample-showcase ranked by flood coverage:", [f"{_tn[k]}({_flood_frac(_tm[k].numpy()):.2f})" for k in range(len(_tn))])
for tag, mdl, fn in [("LiteUNet / FedAvg", lite_fedavg_model, "32_fed_samples_avg.png"),
                     ("LiteUNet / FedProx", lite_fedprox_model, "33_fed_samples_prox.png"),
                     ("LiteUNet / Fed-SalGate", lite_fedsg_model, "34_fed_samples_salgate.png"),
                     ("FSCG-UNet / Fed-SalGate", fscg_fedsg_model, "35_fed_samples_fscgsalgate.png")]:
    plot_sample_grid(mdl, tag, _ti, _tm, _tn, fn)

# cross-model side-by-side: SAME held-out samples, all four models
n = min(4, len(_ti))
fig, axes = plt.subplots(n, 6, figsize=(17, 4.0 * n))
models = [("FedAvg", lite_fedavg_model), ("FedProx", lite_fedprox_model),
          ("Fed-SalGate", lite_fedsg_model), ("FSCG-UNet/Fed-SalGate", fscg_fedsg_model)]
if n == 1: axes = axes[None, :]
for i in range(n):
    axes[i, 0].imshow(_denorm(_ti[i])); axes[i, 0].set_title(f"Image\n{_tn[i]}", fontsize=7); axes[i, 0].axis("off")
    axes[i, 1].imshow(_tm[i].numpy(), cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 1].set_title("GT", fontsize=9); axes[i, 1].axis("off")
    for c, (nm, mdl) in enumerate(models, start=2):
        axes[i, c].imshow(_pred(mdl, _ti[i]), cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1)
        if i == 0: axes[0, c].set_title(nm, fontsize=8)
        axes[i, c].axis("off")
fig.suptitle("Same held-out REAL-test scenes across strategies (FedAvg → Fed-SalGate)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "36_fed_cross_model_samples.png"), dpi=150, bbox_inches="tight"); plt.show()
print("36_fed_cross_model_samples.png saved.")

# confusion matrices per model (held-out REAL test)
@torch.no_grad()
def compute_cm(model, loader):
    model.eval()
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=DEVICE)
    for imgs, msks, _ in loader:
        imgs = imgs.to(DEVICE); msks = msks.to(DEVICE)
        preds = torch.argmax(_forward(model, imgs), dim=1)
        valid = (msks >= 0) & (msks < NUM_CLASSES)
        g = msks[valid]; p = preds[valid]
        cm += torch.bincount(NUM_CLASSES * g + p, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    return cm.cpu().numpy()

cms = {}
for tag, mdl in [("LiteUNet / FedAvg", lite_fedavg_model), ("LiteUNet / FedProx", lite_fedprox_model),
                 ("LiteUNet / Fed-SalGate", lite_fedsg_model), ("FSCG-UNet / Fed-SalGate", fscg_fedsg_model)]:
    cms[tag] = compute_cm(mdl, test_loader)

fig, axes = plt.subplots(2, 2, figsize=(17, 14.5))
for ax, (tag, cm) in zip(axes.ravel(), cms.items()):
    row = cm / np.clip(cm.sum(axis=1, keepdims=True), 1e-9, None)
    im = ax.imshow(row, cmap="Blues", vmin=0, vmax=0.6)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    ax.set_title(f"{tag}  (mIoU {finals[tag]['heldout_miou']:.3f})", fontsize=10)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            if row[r, c] > 0.015:
                ax.text(c, r, f"{row[r, c]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if row[r, c] > 0.35 else "black")
    ax.set_ylabel("Ground truth", fontsize=8); ax.set_xlabel("Predicted", fontsize=8)
plt.suptitle("Per-class confusion matrices — held-out REAL test (rows = GT, cols = Pred)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "37_fed_confusion_matrices.png"), dpi=150, bbox_inches="tight"); plt.show()
print("37_fed_confusion_matrices.png saved.")
cm_arr = np.stack([cms[t] for t in cms], axis=0)                      # [model, GT, pred]
np.save(os.path.join(OUTPUT_DIR, "tables", "p3_confusion_mats.npy"), cm_arr)
with open(os.path.join(OUTPUT_DIR, "tables", "p3_confusion_mats.json"), "w") as _fc:
    json.dump({t: v.tolist() for t, v in cms.items()}, _fc, indent=2)
print("Saved p3_confusion_mats.npy / .json")

# error maps (FP = red, FN = blue) on held-out REAL test
def err_overlay(img, gt, pred):
    rgb = _denorm(img)
    fp = (pred != gt) & (gt == 0)      # background GT but predicted foreground
    fn = (pred == 0) & (gt != 0)       # foreground GT but predicted background
    rgb[fp] = [1.0, 0.2, 0.2]
    rgb[fn] = [0.2, 0.3, 1.0]
    return rgb, fp, fn

for tag, mdl, fn in [("LiteUNet / FedAvg", lite_fedavg_model, "38_fed_errormap_avg.png"),
                     ("LiteUNet / Fed-SalGate", lite_fedsg_model, "39_fed_errormap_salgate.png"),
                     ("FSCG-UNet / Fed-SalGate", fscg_fedsg_model, "40_fed_errormap_fscgsalgate.png")]:
    n = min(4, len(_ti))
    fig, axes = plt.subplots(n, 4, figsize=(15, 4.2 * n))
    if n == 1: axes = axes[None, :]
    for i in range(n):
        gt = _tm[i].numpy(); pred = _pred(mdl, _ti[i])
        ov, fp, fnm = err_overlay(_ti[i], gt, pred)
        axes[i, 0].imshow(_denorm(_ti[i])); axes[i, 0].set_title(f"Image - {_tn[i]}", fontsize=8); axes[i, 0].axis("off")
        axes[i, 1].imshow(gt, cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 1].set_title("GT", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(pred, cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1); axes[i, 2].set_title("Prediction", fontsize=9); axes[i, 2].axis("off")
        axes[i, 3].imshow(ov); axes[i, 3].set_title("FP / FN", fontsize=9); axes[i, 3].axis("off")
    fig.suptitle(f"{tag} — error maps (red = false positive, blue = false negative)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fn), dpi=150, bbox_inches="tight"); plt.show()
    print(fn, "saved.")

# saliency-gate visual: the actual maps feeding the consensus (final-round snapshot)
for tag, hist, fn in [("Lite-UNet / Fed-SalGate", hist_fedsg, "41_fed_saliency_gates.png"),
                      ("FSCG-UNet / Fed-SalGate", hist_fscg_sg, "42_fed_saliency_gates_big.png")]:
    snap = hist.get("snapshot")
    if not snap:
        print(f"no snapshot for {tag}"); continue
    si, names = snap["imgs"], snap["names"]
    pc = np.array(snap["per_client"])            # [clients, probe]
    cons = np.array(snap["consensus"]) if snap["consensus"] is not None else pc.mean(0)
    gates, agrs = np.array(snap["gates"]), np.array(snap["agreements"])
    npr = min(4, len(si))
    fig, axes = plt.subplots(npr, NUM_CLIENTS + 2, figsize=(2.4 * (NUM_CLIENTS + 2), 2.6 * npr))
    for i in range(npr):
        axes[i, 0].imshow(_denorm(si[i])); axes[i, 0].set_title(f"Probe {names[i]}", fontsize=7); axes[i, 0].axis("off")
        axes[i, 1].imshow(cons[i], cmap="hot_r"); axes[i, 1].set_title("Consensus", fontsize=8); axes[i, 1].axis("off")
        for c in range(NUM_CLIENTS):
            axes[i, c + 2].imshow(_denorm(si[i]), alpha=0.35); axes[i, c + 2].imshow(pc[c][i], cmap="hot_r", alpha=0.65)
            if i == 0:
                axes[0, c + 2].set_title(f"C{c} (λ={gates[c]:.2f}, agg={agrs[c]:.2f})", fontsize=7)
            axes[i, c + 2].axis("off")
    fig.suptitle(f"{tag} — saliency maps feeding the aggregation gate (λ=gate weight, agg=agreement vs consensus)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, fn), dpi=150, bbox_inches="tight"); plt.show()
    print(fn, "saved.")

print("Qualitative validation figures complete.")

## 💾 Save Part-3 outputs

In [ ]:

# =====================================================================
# CRASH-PROOF SAVE BLOCK
#  - writes PART3_SUMMARY.txt + part3_early_summary.json FIRST (before any
#    heavy serialization), using a num/jnp/torch-safe serializer
#  - every artifact (json/csv/png is written inside try/except so one
#    failing file can never abort the whole save
#  - zip is produced AFTERward, per-file with try/except
# =====================================================================
import zipfile

# numeric-dtype-safe serializer (numpy scalars, 0-d/1-d arrays, tensors)
def _surrogate(o):
    if isinstance(o, dict):
        return {str(k): _surrogate(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_surrogate(v) for v in o]
    if isinstance(o, (np.ndarray,)):
        return _surrogate(o.tolist())
    if o is None or isinstance(o, (bool, str)):
        return o
    if isinstance(o, (int, float)):
        return o
    try:
        if hasattr(o, "item"):                 # torch / np scalar
            return _surrogate(o.item())
    except Exception:
        pass
    try:
        return float(o)
    except (TypeError, ValueError, OverflowError):
        try:
            return str(o)
        except Exception:
            return "<unserializable>"

def _atomic_json(obj, path):
    """Serialize & write atomically; never raises."""
    try:
        s = json.dumps(obj, indent=2, default=_surrogate)
        _fd, _tmp = tempfile.mkstemp(prefix=".at_", dir=os.path.dirname(path) or ".")
        with os.fdopen(_fd, "w") as f:
            f.write(s)
        os.replace(_tmp, path)
        return f"OK {os.path.basename(path)} ({len(s)//1024} KB)"
    except Exception as e:
        return f"FAIL {os.path.basename(path)}: {type(e).__name__}: {e}"

def _safe_txt(txt, path):
    try:
        with open(path, "w") as f:
            f.write("\n".join(txt))
        return f"OK {os.path.basename(path)}"
    except Exception as e:
        return f"FAIL {os.path.basename(path)}: {type(e).__name__}: {e}"

# ---- summarize main quartet drawn EXCLUSIVELY from finals (saved in eval cell) ----
model_tags = ["LiteUNet / FedAvg", "LiteUNet / FedProx", "LiteUNet / Fed-SalGate",
              "FSCG-UNet / Fed-SalGate"]
txt = [
    "Fed-SALGATE Part 3 -- Federated Results (held-out REAL GT)",
    "=" * 58,
]
if finals:
    for tag in model_tags:
        f = finals[tag]
        txt.append(f"{tag:<30} mIoU={f['heldout_miou']:.4f}  pixel_acc={f['pixel_acc']:.4f}")
    if "deltas_main" in globals():
        d = deltas_main
        txt += [
            "-" * 58,
            f"Fed-SalGate gain vs FedAvg (LiteUNet): {d['vs_fedavg']:+.4f}",
            f"Fed-SalGate gain vs FedProx (LiteUNet): {d['vs_fedprox']:+.4f}",
            f"FSCG-UNet/FedSalGate vs LiteUNet/FedAvg: {d['fscg_vs_avg']:+.4f}",
        ]
if seed_finals:
    txt += ["-" * 58,
            f"SEED STUDY (n={seed_stats['n_seeds']}): FedAvg={seed_stats['fedavg_mean']:.4f}±{seed_stats['fedavg_std']:.4f}  "
            f"Fed-SalGate={seed_stats['fedsalgate_mean']:.4f}±{seed_stats['fedsalgate_std']:.4f}  Δ={seed_stats['delta_mean']:+.4f}"]
if robust_stats:
    txt += ["-" * 58,
            f"ROBUSTNESS (client {robust_stats['degraded_client']} degraded p={robust_stats['degrade_p']:.2f}): "
            f"FedAvg={robust_stats['fedavg_miou']:.4f}  Fed-SalGate={robust_stats['fedsalgate_miou']:.4f}  "
            f"margin={robust_stats['margin']:+.4f}"]
summary_path = os.path.join(OUTPUT_DIR, "PART3_SUMMARY.txt")
print(_safe_txt(txt, summary_path))
print("  -> ", summary_path)

# ---- aggregate results JSON (single dict, numeric-safe) ----
result = {
    "num_clients": NUM_CLIENTS,
    "num_rounds": NUM_ROUNDS,
    "local_epochs": FL_LOCAL_EPOCHS,
    "client_subsample": CLIENT_SUBSAMPLE,
    "gate_beta": GATE_BETA,
    "gate_tau": GATE_TAU,
    "gate_ema": GATE_EMA,
    "probe_k": PROBE_K,
    "saliency_classes": [int(c) for c in SAL_CLASSES],
    "config": {"alpha": DIRICHLET_ALPHA, "img_size": IMG_SIZE, "batch": BATCH_SIZE, "lr": LR,
               "amp": bool(AMP)},
}
if finals:
    result["models"] = {tag: {"heldout_miou": f["heldout_miou"],
                              "pixel_acc": f["pixel_acc"],
                              "best_round_miou": f.get("best_round_miou", f["heldout_miou"]),
                              "per_class_iou": {str(k): round(float(v), 4)
                                                for k, v in f["per_class"].items()}}
                        for tag, f in finals.items()}
if "deltas_main" in globals():
    result["deltas"] = dict(deltas_main)
if seed_finals:
    result["seed_study"] = seed_stats
    result["seed_rows"] = [{k: r[k] for k in ("agg", "seed", "miou", "pix")} for r in seed_finals]
if robust_stats:
    result["robustness"] = robust_stats

early_path = os.path.join(OUTPUT_DIR, "part3_early_summary.json")
print(_atomic_json(result, early_path))                                  # early, light
result_path = os.path.join(OUTPUT_DIR, "part3_fedsalgate_results.json")
print(_atomic_json(result, result_path))                                 # full (same dict)

# ---- CSV exports (numeric-safe, try/except per frame) ----
try:
    if finals:
        rows = []
        for tag in model_tags:
            f = finals[tag]
            rows.append({"model": tag, "heldout_miou": f["heldout_miou"],
                         "pixel_acc": f["pixel_acc"],
                         **{f"iou_{CLASS_NAMES[c]}": f["per_class"].get(c, 0.0)
                            for c in range(NUM_CLASSES)}})
        pd.DataFrame(rows).set_index("model").to_csv(
            os.path.join(OUTPUT_DIR, "tables", "p3_fed_per_class_iou.csv"))
        print("OK p3_fed_per_class_iou.csv")
    if seed_finals:
        pd.DataFrame(seed_finals).to_csv(
            os.path.join(OUTPUT_DIR, "tables", "p3_fed_seed_study.csv"), index=False)
        print("OK p3_fed_seed_study.csv")
    if robust_stats:
        pd.DataFrame([robust_stats]).to_csv(
            os.path.join(OUTPUT_DIR, "tables", "p3_fed_robustness.csv"), index=False)
        print("OK p3_fed_robustness.csv")
except Exception as _e:
    print("CSV export issue (non-fatal):", type(_e).__name__, _e)

# ---- zip OUTPUTS (never checkpoint .pth, never resized_cache; per-file try) ----
_zip_path = os.path.join(WORKING_DIR, "part3_outputs.zip")
_ok, _skip = 0, 0
try:
    with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for _root, _dirs, _files in os.walk(OUTPUT_DIR):
            for _f in _files:
                if _f.endswith(".pth") or _root.endswith("resized_cache"):
                    _skip += 1
                    continue
                try:
                    z.write(os.path.join(_root, _f),
                            os.path.relpath(os.path.join(_root, _f), WORKING_DIR))
                    _ok += 1
                except Exception:
                    _skip += 1
    print(f"Zipped part3_outputs.zip -> {_zip_path}  ({os.path.getsize(_zip_path)/1e6:.2f} MB, "
          f"{_ok} files, skipped {_skip})")
except Exception as _e:
    print("ZIP FAILED but save already complete:", type(_e).__name__, _e)

print("\nDONE -- Part 3 finished. All Results saved under fed_salgate_outputs.")
